# CIS Benchmark Compliance Checks

> This is a Jupyter Notebook designed to run in [Visual Studio Code](https://code.visualstudio.com/) with the [Jupyter extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter), or in [JupyterLab](https://jupyterlab.readthedocs.io/en/latest/).
>
> This notebook uses the **PowerShell kernel** and requires the [dbatools](https://dbatools.io/) and [kbupdate](https://github.com/potatoqualitee/kbupdate) modules.

This notebook checks CIS Benchmark compliance for SQL Server instances. The checks are derived from the **CIS Microsoft SQL Server 2022 Benchmark v1.2.1**, available at [cisecurity.org](https://www.cisecurity.org/).

Each section includes the benchmark description, rationale, and both T-SQL reference queries and dbatools PowerShell audit commands. Remediation commands are provided but commented out by default to prevent accidental changes.

**Conventions:**
- **CIS Benchmark Query (T-SQL Reference):** — The audit query from the CIS benchmark document, shown for reference.
- **dbatools PowerShell Command:** — The executable equivalent using dbatools cmdlets.

## **Specify SQL Servers**

List out which SQL Server Instances will need to be patched. For multiple server seperate with a comma.

In [ ]:
$SqlInstances = [System.Collections.ArrayList]@();

# Add your SQL Server instances here
# $SqlInstances.Add([pscustomobject]@{Instance = "YOURSERVER01"}) | Out-Null;
# $SqlInstances.Add([pscustomobject]@{Instance = "YOURSERVER02"}) | Out-Null;

# Display the list of servers
$SqlInstances |
Format-Markdown

---

# 1. Installation, Updates and Patches

## 1.1. Check The Current Patch Level (Level 1)
> **Benchmark Description:**
>
>SQL Server patches contain program updates that fix security and product functionality issues found in the software. These patches can be installed with a hotfix which is a single patch, a cumulative update which is a small group of patches or a service pack which is a large collection of patches. The SQL Server version and patch levels should be the most recent compatible with the organizations' operational needs.

> **Benchmark Rationale:**
>
> Using the most recent SQL Server software, along with all applicable patches can help limit the possibilities for vulnerabilities in the software. The installation version and/or patches applied during setup should be established according to the needs of the organization.

### **Audit:**

This step will retrieve the current patch level of the SQL Instances in comparison to the latest patch that is available.

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT SERVERPROPERTY('ProductLevel')           as [SP_installed],
       SERVERPROPERTY('ProductVersion')         as [Version],
       SERVERPROPERTY('ProductUpdateLevel')     as [ProductUpdate_Level],
       SERVERPROPERTY('ProductUpdateReference') as [KB_Number];
```

**dbatools PowerShell Command:**

In [ ]:
# Check what servers need to be patched

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Update = $true
    #MaxBehind = '0CU'
    Latest = $true
};

try {
    # Capture patch level of each server
#    Test-DbaBuild @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue | 
#    Select-Object SqlInstance, NameLevel, KBLevel, SPLevel, CULevel, SPTarget, CUTarget, BuildLevel, BuildTarget, Compliant |
    # Replace empty values with a space
#    ForEach-Object {$_.PSObject.Properties | ForEach-Object {if (-not $_.Value) { $_.Value = " " }} $_}    
    #Format-Table -Property * -AutoSize;
#    Format-Markdown

    # Capture patch level of each server
    $results = Test-DbaBuild @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue | 
    Select-Object SqlInstance, NameLevel, KBLevel, SPLevel, CULevel, SPTarget, CUTarget, BuildLevel, BuildTarget, Compliant

    # Ensure empty values are replaced
    $results | ForEach-Object {
        $_ | ForEach-Object {
            $_.PSObject.Properties | ForEach-Object {
                if (-not $_.Value) { $_.Value = " " } # Replace empty values with a space
            }
        }
        $_
    } | Format-Markdown
}
catch { throw $_ }

## 1.2. Ensure Single-Function Member Server Use

> **Benchmark Description:**
>
>It is recommended that SQL Server software be installed on a dedicated server. This architectural consideration affords security flexibility in that the database server can be placed on a separate subnet allowing access only from particular hosts and over particular protocols. Degrees of availability are easier to achieve as well - over time, an enterprise can move from a single database server to a failover to a cluster using load balancing or to some combination thereof.


> **Benchmark Rationale:**
>
> It is easier to manage (i.e. reduce) the attack surface of the server hosting SQL Server software if the only surfaces to consider are the underlying operating system, SQL Server itself, and any security/operational tooling that may additionally be installed. As noted in the description, availability can be more easily addressed if the database is on a dedicated server.

### **Audit:**

Verify that production SQL Server instances are hosted on dedicated servers or cluster nodes, not shared with other application workloads. Document your environment's server topology and separation of duties between infrastructure and database administration.

---

# 2. Surface Area Reduction

> SQL Server offers various configuration options, some of them can be controlled by the sp\_configure stored procedure. This section contains the listing of the corresponding recommendations.

## 2.1 Ensure 'Ad Hoc Distributed Queries' Server Configuration Option if set to '0' (Automated)

> **Benchmark Description:**
>
>Enabling ***Ad Hoc Distributed Queries*** allows users to query data and execute statements on external data sources. This functionality should be disabled.

> **Benchmark Rationale:**
>
> This feature can be used to remotely access and exploit vulnerabilities on remote SQL Server instances and to run unsafe Visual Basic for Application functions.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**
```sql
-- Check the current state of the option
SELECT name, CAST(value as int) as value_configured, CAST(value_in_use as int) as value_in_use 
FROM sys.configurations 
WHERE name = 'Ad Hoc Distributed Queries';

-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1; 
RECONFIGURE; 
GO
EXECUTE sp_configure 'Ad Hoc Distributed Queries', 0; 
RECONFIGURE; 
GO 
EXECUTE sp_configure 'show advanced options', 0; 
RECONFIGURE;
GO
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Ad Hoc Distributed Queries' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'AdHocDistributedQueriesEnabled'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

In [ ]:
# UPDATE THE 'Ad Hoc Distributed Queries' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'AdHocDistributedQueriesEnabled'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.2 Ensure 'CLR Enabled' Server Configuration Option is set to '0' (Automated)

> **Benchmark Description:**
> 
> The _**clr enabled**_ option specifies whether user assemblies can be run by SQL Server.
> **Benchmark Rationale:**
> 
> Enabling use of CLR assemblies widens the attack surface of SQL Server and puts it at risk from both inadvertent and malicious assemblies.
> **Possible Impact:** If CLR assemblies are in use, applications may need to be re-architected to eliminate their usage before disabling this setting. Alternatively, some organizations may allow this setting to be enabled 1 for assemblies created with the SAFE permission set, but disallow assemblies created with the riskier UNSAFE and EXTERNAL\_ACCESS permission sets. To find user-created assemblies, run the following query in all databases, replacing with each database name:
```
USE [<database_name>] 
GO 
SELECT name AS Assembly_Name, permission_set_desc 
FROM sys.assemblies 
WHERE is_user_defined = 1; 
GO
```

<u><mark>**NOTE:** Review your environment for databases that depend on CLR assemblies before disabling this option.</mark></u>

### **Audit:**
**CIS Benchmark Query (T-SQL Reference):**
```

-- SQL 2022
SELECT 
    name, 
    CAST(value as int) as value_configured, 
    CAST(value_in_use as int) as value_in_use 
FROM sys.configurations
WHERE name = 'clr strict security';

SELECT 
    name, 
    CAST(value as int) as value_configured, 
    CAST(value_in_use as int) as value_in_use 
FROM sys.configurations 
WHERE name = 'clr enabled';
GO
-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1; 
RECONFIGURE; 
GO
EXECUTE sp_configure 'clr enabled', 0; 
RECONFIGURE; 
GO 
EXECUTE sp_configure 'show advanced options', 0; 
RECONFIGURE;
GO
```
**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'CLR Strict Security' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'ClrStrictSecurity'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

In [ ]:
# CHECK THE CURRENT 'CLR Enabled' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'IsSqlClrEnabled'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

<u><mark>**WARNING: POTENTIAL BREAKING CHANGE.** Review your environment for databases that depend on CLR assemblies before disabling. Disabling CLR without verifying dependencies will break functionality.</mark></u>

In [ ]:
# UPDATE THE 'CLR Enabled' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'IsSqlClrEnabled'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.3 Ensure 'Cross DB Ownership Chaining' Server Configuration Option if set to '0' (Automated)

> **Benchmark Description:**
>
> The ***cross db ownership chaining*** option controls cross-database ownership chaining across all databases at the instance (or server) level.

> **Benchmark Rationale:**
>
> When enabled, this option allows a member of the _db_owner_ role in a database to gain access to objects owned by a login in any other database, causing an unnecessary information disclosure. When required, cross-database ownership chaining should only be enabled for the specific databases requiring it instead of at the instance level for all databases by using the `ALTER DATABASE<database_name > SET DB_CHAINING ON` command. This database option may not be changed on the _master_, _model_, or _tempdb_ system databases.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**
```sql
-- Check the current value
SELECT 
    name, 
    CAST(value as int) as value_configured, 
    CAST(value_in_use as int) as value_in_use 
FROM sys.configurations 
WHERE name = 'cross db ownership chaining';
GO

-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1; 
RECONFIGURE; 
GO
EXECUTE sp_configure 'cross db ownership chaining', 0; 
RECONFIGURE; 
GO 
EXECUTE sp_configure 'show advanced options', 0; 
RECONFIGURE;
GO
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Cross DB Ownership Chaining' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'CrossDBOwnershipChaining'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

In [ ]:
# UPDATE THE 'Cross DB Ownership Chaining' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'CrossDBOwnershipChaining'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.4 Ensure 'Database Mail XPs' Server Configuration Option if set to '0' (Automated)

> **Benchmark Description:**
>
> The ***Database Mail XPs*** option specifies whether user assemblies can be run by SQL Server.

> **Benchmark Rationale:**
>
> Disabling the ***Database Mail XPs*** option reduces the SQL Server surface, eliminates a DOS attack vector and channel to exfiltrate data from the database server to a remote host.

> **Possible Impact:**
>
> By disabling this option, the server will no longer be able to send alerts for SQL Agent Job failures.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**
```sql
-- Check the current value
SELECT 
    name, 
    CAST(value as int) as value_configured, 
    CAST(value_in_use as int) as value_in_use 
FROM sys.configurations 
WHERE name = 'Database Mail XPs';
GO

-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1; 
RECONFIGURE; 
GO
EXECUTE sp_configure 'Database Mail XPs', 0; 
RECONFIGURE; 
GO 
EXECUTE sp_configure 'show advanced options', 0; 
RECONFIGURE;
GO
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Database Mail XPs' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'DatabaseMailEnabled'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

<u><mark>**WARNING:** If your environment uses Database Mail for SQL Agent job failure alerts or other event notifications, disabling this option will break that functionality.</mark></u>

In [ ]:
# UPDATE THE 'Database Mail XPs' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'DatabaseMailEnabled'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.5 Ensure 'Ole Automation Procedures' Server Configuration Option if set to '0' (Automated)

> **Benchmark Description:**
>
> The ***Ole Automation Procedures*** option controls whether OLE Automation objects can be instantiated within Transact-SQL batches. These are extended stored procedures that allow SQL Server users to execute functions external to SQL Server.

> **Benchmark Rationale:**
>
> Enabling this option will increase the attack surface of SQL Server and allow users to execute functions in the security context of SQL Server.

> **Possible Impact:**
>
> Disabling this option could impact third-party monitoring tools that rely on OLE Automation for data collection.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**
```sql
-- Check the current value
SELECT 
    name, 
    CAST(value as int) as value_configured, 
    CAST(value_in_use as int) as value_in_use 
FROM sys.configurations 
WHERE name = 'Ole Automation Procedures';
GO

-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1; 
RECONFIGURE; 
GO
EXECUTE sp_configure 'Ole Automation Procedures', 0; 
RECONFIGURE; 
GO 
EXECUTE sp_configure 'show advanced options', 0; 
RECONFIGURE;
GO
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Ole Automation Procedures' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'OleAutomationProceduresEnabled'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

<u><mark>**WARNING:** If your environment uses third-party monitoring tools that rely on OLE Automation (e.g., Idera SQL Diagnostic Manager), verify compatibility before disabling this option.</mark></u>

In [ ]:
# UPDATE THE 'Ole Automation Procedures' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'OleAutomationProceduresEnabled'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.6 Ensure 'Remote Access' Server Configuration Option if set to '0' (Automated)
> **Benchmark Description:**
> 
> The _**remote access**_ option controls the execution of local stored procedures on remote servers or remote stored procedures on local server.
> **Benchmark Rationale:**
> 
> Functionality can be abused to launch a Denial-of-Service (DoS) attack on remote servers by off-loading query processing to a target.
> **Possible Impact:**
> 
> Per Microsoft: This feature will be removed in the next version of Microsoft SQL Server. Do not use this feature in new development work, and modify applications that currently use this feature as soon as possible. Use _sp\_addlinkedserver_ instead.
### **Audit:**
**CIS Benchmark Query (T-SQL Reference):**
```
-- Check the current value
SELECT 
    name, 
    CAST(value as int) as value_configured, 
    CAST(value_in_use as int) as value_in_use 
FROM sys.configurations 
WHERE name = 'remote access';
GO
-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1; 
RECONFIGURE; 
GO
EXECUTE sp_configure 'remote access', 0; 
RECONFIGURE; 
GO 
EXECUTE sp_configure 'show advanced options', 0; 
RECONFIGURE;
GO
```
**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'remote access' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'RemoteAccess'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

In [ ]:
# UPDATE THE 'remote access' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'RemoteAccess'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.7 Ensure 'Remote Admin Connections' Server Configuration Option if set to '0' (Automated)

> **Benchmark Description:**
>
> The ***remote admin connections*** option controls whether a client application on a remote computer can use the Dedicated Administrator Connection (DAC).

> **Benchmark Rationale:**
>
> The Dedicated Administrator Connection (DAC) lets an administrator access a running server to execute diagnostic functions or Transact-SQL statements, or to troubleshoot problems on the server, even when the server is locked or running in an abnormal state and not responding to a SQL Server Database Engine connection. In a cluster scenario, the administrator may not actually be logged on to the same node that is currently hosting the SQL Server instance and thus is considered "remote". Therefore, this setting should usually be enabled (1) for SQL Server failover clusters; otherwise, it should be disabled (0) which is the default.

> **Possible Impact:**
>
> Disabling this option could impact the ability to access as locked up server, especially if we are not able to login to the effected server or cluster node.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**
```sql
-- Check the current value
SELECT 
    name, 
    CAST(value as int) as value_configured, 
    CAST(value_in_use as int) as value_in_use 
FROM sys.configurations 
WHERE name = 'remote admin connections'
AND SERVERPROPERTY('IsClustered') = 0;
GO

-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1; 
RECONFIGURE; 
GO
EXECUTE sp_configure 'remote admin connections', 0; 
RECONFIGURE; 
GO 
EXECUTE sp_configure 'show advanced options', 0; 
RECONFIGURE;
GO
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Remote Admin Connections' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'RemoteDacConnectionsEnabled'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

<u><mark>**WARNING:** If your environment uses Windows Failover Clustering, disabling Remote Admin Connections could prevent you from connecting to a locked instance on a node you cannot directly log into. Consider leaving this enabled (1) for clustered instances.</mark></u>

In [ ]:
# UPDATE THE 'Remote Admin Connections' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'RemoteDacConnectionsEnabled'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.8 Ensure 'Scan For Startup Procs' Server Configuration Option if set to '0' (Automated)

> **Benchmark Description:**
>
> The scan for startup procs option, if enabled, causes SQL Server to scan for and automatically run all stored procedures that are set to execute upon service startup.

> **Benchmark Rationale:**
>
> Enforcing this control reduces the threat of an entity leveraging these facilities for malicious purposes.

> **Possible Impact:**
>
> Setting Scan for Startup Procedures to 0 will prevent certain audit traces and other commonly used monitoring stored procedures from re-starting on start up. Additionally, replication requires this setting to be enabled (1) and will automatically change this setting if needed.

### Audit:

**CIS Benchmark Query (T-SQL Reference):**

```sql
-- Check the current value
SELECT
    name,
    CAST(value as int) as value_configured,
    CAST(value_in_use as int) as value_in_use
FROM sys.configurations
WHERE name = 'scan for startup procs';
GO

-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1;
RECONFIGURE;
GO
EXECUTE  sp_configure  'scan for startup procs',  0;
RECONFIGURE;
GO
EXECUTE sp_configure 'show advanced options', 0;
RECONFIGURE;
GO
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Scan For Startup Procs' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'ScanForStartupProcedures'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

<u><mark>**WARNING:** If your environment uses Transactional Replication, disabling Scan for Startup Procedures should be done with care. Replication requires this setting and will automatically re-enable it if needed.</mark></u>

In [ ]:
# UPDATE THE 'Scan For Startup Procs' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'ScanForStartupProcedures'
};

try {
    # Set the Advanced Option to False
    #Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    #Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

## 2.9 Ensure 'Trustworthy' Server Configuration Option if set to '0' (Automated)

> **Benchmark Description:**
>
> The ***TRUSTWORTHY*** database option allows database objects to access objects in other databases under certain circumstances.

> **Benchmark Rationale:**
>
> Provides protection from malicious CLR assemblies or extended procedures.

> **Possible Impact:**
>
> 

### Audit:

**CIS Benchmark Query (T-SQL Reference):**

```sql
-- Check the current value
SELECT 
    name 
FROM sys.databases 
WHERE is_trustworthy_on = 1 
AND name != 'msdb';
GO

-- Set the state of the option
ALTER DATABASE [<database_name>] SET TRUSTWORTHY OFF;
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Trustworthy' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Advanced Option current values
    Get-DbaDatabase -sqlinstance $instances.SqlInstance -WarningAction SilentlyContinue | 
    Where-Object { ($_.Name -ne 'msdb') } |
    Select-Object SqlInstance, Name, Trustworthy | 
    Sort-Object -Property @{Expression = "Trustworthy"; Descending = $true }, @{Expression = "SqlInstance"; Descending = $false }, @{Expression = "Name"; Descending = $false } |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

## 2.10 Ensure Unnecessary SQL Server Protocols are set to 'Disabled' (Manual)

> **Benchmark Description:**
>
> SQL Server supports Shared Memory, Named Pipes, and TCP/IP protocols. However, SQL Server should be configured to use the bare minimum required based on the organization's needs.

> **Benchmark Rationale:**
>
> Using fewer protocols minimizes the attack surface of SQL Server and, in some cases, can protect it from remote attacks.

> **Possible Impact:**
>
> The Database Engine (MSSQL and SQLAgent) services must be stopped and restarted for the change to take effect.

### Audit:

Open **SQL Server Configuration Manager**; go to the **SQL Server Network Configuration**. Ensure that only required protocols are enabled.

**Default Value:**
By default, TCP/IP and Shared Memory protocols are enabled on all commercial editions.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT ENABLED PROTOCOLS

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Advanced Option current values
    Get-DbaInstanceProtocol -ComputerName $instances.SqlInstance -WarningAction SilentlyContinue | 
    Select-Object ComputerName, DisplayName, Order, IsEnabled | 
    #Format-Table -AutoSize;
    Format-Markdown
}
catch { throw $_ }

## 2.11 Ensure SQL Server is configured to use non-standard ports (Automated)

> **Benchmark Description:**
>
> If installed, a default SQL Server instance will be assigned a default port of _TCP:1433_ for TCP/IP communication. Administrators can also manually configure named instances to use _TCP:1433_ for communication. _TCP:1433_ is a widely known SQL Server port and this port assignment should be changed. In a multi-instance scenario, each instance must be assigned its own dedicated TCP/IP port.

> **Benchmark Rationale:**
>
> Using a non-default port helps protect the database from attacks directed to the default port.

> **Possible Impact:**
>
> Changing the default port will force the DAC (Dedicated Administrator Connection) to listen on a random port. Also, it might make benign applications, such as application firewalls, require special configuration. In general, you should set a static port for consistent usage by applications, including firewalls, instead of using dynamic ports which will be chosen randomly at each SQL Server start up.

### Audit:

Run the following T-SQL script:

> A value of **0** implies a pass.

```sql
SELECT count(1) 
FROM sys.dm_server_registry 
WHERE value_name like '%Tcp%' and value_data='1433';
```

**Default Value:**
By default, default SQL Server instances listen on to TCP/IP traffic on TCP port 1433 and named instances use dynamic ports.

### Remediation:
1. In **SQL Server Configuration Manager**, in the console pane, expand **SQL Server Network Configuration**, expand Protocols for _<InstanceName>_, and then double-click the TCP/IP protocol
2. In the **TCP/IP Properties** dialog box, on the **IP Addresses** tab, several IP addresses appear in the format _IP1_, _IP2_, up to _IPAll_. One of these is for the IP address of the loopback adapter, _127.0.0.1_. Additional IP addresses appear for each IP Address on the computer.
Page 39
3. Under _IPAll_, change the **TCP Port** field from _1433_ to a non-standard port or leave the **TCP Port** field empty and set the **TCP Dynamic Ports** value to 0__ to enable dynamic port assignment and then click **OK**.
4. In the console pane, click **SQL Server Services**.
5. In the details pane, right-click **SQL Server (_<InstanceName>_)** and then click **Restart**, to stop and restart SQL Server.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT TCP PORT NUMBER

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Advanced Option current values
    Get-DbaTcpPort -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue | 
    Select-Object SqlInstance, IPAddress, Port, Static | 
    #Format-Table -AutoSize;
    Format-Markdown
}
catch { throw $_ }

<mark>**WARNING!:** This will cause a restart of the SQL services. For the clusters this should be done manually using the steps outlined under Remediation and the cluster instances manually failed over. This will also require significant reconfigurations with the company applications.</mark>

In [ ]:
# UPDATE THE TCP PORT

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Port    = 1433
    Confirm = $false
};

try {
    # Set the Advanced Option to False
    Set-DbaTcpPort -SqlInstance $instances.SqlInstance @params -WarningAction SilentlyContinue |
    Format-Table -AutoSize;

    # Capture Advanced Option current values
    Get-DbaTcpPort -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue | 
    Select-Object SqlInstance, IPAddress, Port, Static | 
    Format-Table -AutoSize;
}
catch { throw $_ }

## 2.12 Ensure 'Hide Instance' option is set to 'Yes' for Production SQL Server Instances (Automated)

> **Benchmark Description:**
>
> Non-clustered SQL Server instances within production environments should be designated as hidden to prevent advertisement by the SQL Server Browser service.

> **Benchmark Rationale:**
>
> Designating production SQL Server instances as hidden leads to a more secure installation because they cannot be enumerated. However, clustered instances may break if this option is selected.

> **Possible Impact:**
>
> This method only prevents the instance from being listed on the network. If the instance is hidden (not exposed by SQL Browser), then connections will need to specify the server and port in order to connect. It does not prevent users from connecting to server if they know the instance name and port.
>
> <mark>**WARNING!:** If you hide a clustered named instance, the cluster service may not be able to connect to the SQL Server. Please refer to the Microsoft documentation reference.</mark>

### Audit:

**GUI Method:**
1. In **SQL Server Configuration Manager**, expand **SQL Server Network Configuration**, right-click **Protocols for _< InstanceName >_**, and then select **Properties**.
2. On the **Flags** tab, in the **Hide Instance** box, if _Yes_ is selected, it is compliant.

**CIS Benchmark Query (T-SQL Reference):**

> A value of **1** implies a compliance.

```sql
DECLARE @getValue INT; 
EXEC master.sys.xp_instance_regread 
    @rootkey = N'HKEY_LOCAL_MACHINE', 
    @key = N'SOFTWARE\Microsoft\Microsoft SQL Server\MSSQLServer\SuperSocketNetLib', 
    @value_name = N'HideInstance', 
    @value = @getValue OUTPUT; 
SELECT @getValue;
```

**Default Value:**
By default, SQL Server instances are not hidden.

### Remediation:
**GUI Method**
1. In **SQL Server Configuration Manager**, expand **SQL Server Network Configuration**, right-click **Protocols for _< InstanceName >_**, and then select **Properties**.
2. On the **Flags** tab, in the **Hide Instance** box, select _Yes_, and then click **OK** to close the dialog box. The change takes effect immediately for new connections.

**CIS Benchmark Query (T-SQL Reference):**
```sql
EXEC master.sys.xp_instance_regwrite 
    @rootkey = N'HKEY_LOCAL_MACHINE',
    @key = N'SOFTWARE\Microsoft\Microsoft SQL Server\MSSQLServer\SuperSocketNetLib', 
    @value_name = N'HideInstance', 
    @type = N'REG;
```

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Hide Instance' VALUE

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaHideInstance -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, HideInstance |
    Format-Markdown
}
catch { throw $_ }

## 2.13 Ensure the 'sa' Login Account is set to 'Disabled' (Automated)

> **Benchmark Description:**
>
> The sa account is a widely known and often widely used SQL Server account with sysadmin privileges. This is the original login created during installation and always has the _principal_id=1_ and _sid=0x01_.

> **Benchmark Rationale:**
>
> Enforcing this control reduces the probability of an attacker executing brute force attacks against a well-known principal.

> **Possible Impact:**
>
> It is not a good security practice to code applications or scripts to use the _sa_ account. However, if this has been done, disabling the _sa_ account will prevent scripts and applications from authenticating to the database server and executing required tasks or functions.

### Audit:

Use the following syntax to determine if the _sa_ account is disabled. Checking for _sid=0x01_ ensures that the original _sa_ account is being checked in case it has been renamed per best practices.

**CIS Benchmark Query (T-SQL Reference):**

> No rows should be returned to be compliant. An _is_disabled_ value of _0_ indicates the login is currently enabled and therefore needs remediation.

```sql
SELECT name, is_disabled
FROM sys.server_principals
WHERE sid = 0x01
AND is_disabled = 0;
```

**Default Value:**

By default, the _sa_ login account is disabled at install time when Windows Authentication Mode is selected. If mixed mode (SQL Server and Windows Authentication) is selected at install, the default for the _sa_ login is enabled.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE 'sa' ACCOUNT

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Login current values
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Login 'sa' -Detailed | 
    #Select-Object SqlInstance, Name, SidString, CreateDate, DateLastModified, LastLogin, BadPasswordCount, BadPasswordTime, PasswordLastSetTime, HasAccess | 
    Select-Object SqlInstance, Name, SidString, HasAccess, IsLocked, IsDisabled | 
    #Format-Table -AutoSize;
    Format-Markdown

}
catch { throw $_ }

### Remediation:

**CIS Benchmark Query (T-SQL Reference):**
```sql
USE [master] 
GO 
ALTER LOGIN sa DISABLE
GO
```

**dbatools PowerShell Command:**

In [ ]:
# UPDATE THE 'sa' ACCOUNT

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Set the Login to Disabled
    Set-DbaLogin -SqlInstance $instances.SqlInstance -Login 'sa' -Disable -WarningAction SilentlyContinue |
    Format-Table -AutoSize;

    # Capture Login current values
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Login 'sa' -Detailed | 
    Select-Object SqlInstance, Name, SidString, CreateDate, DateLastModified, LastLogin, BadPasswordCount, BadPasswordTime, PasswordLastSetTime, HasAccess | 
    Format-Table -AutoSize;
}
catch { throw $_ }

## 2.14 Ensure the 'sa' Login Account has been renamed (Automated)

> **Benchmark Description:**
>
> The sa account is a widely known and often widely used SQL Server account with sysadmin privileges. This is the original login created during installation and always has the _principal_id=1_ and _sid=0x01_.

> **Benchmark Rationale:**
>
> It is more difficult to launch password-guessing and brute-force attacks against the _sa_ login if the name is not known.

> **Possible Impact:**
>
> It is not a good security practice to code applications or scripts to use the _sa_ account. However, if this has been done, disabling the _sa_ account will prevent scripts and applications from authenticating to the database server and executing required tasks or functions.

### Audit:

Use the following syntax to determine if the _sa_ login (principal) is renamed.

> A name of _sa_ indicates the account has not been renamed and therefore needs remediation.

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT name
FROM sys.server_principals
WHERE sid = 0x01;
```

**Default Value:**

By default, the _sa_ login name is 'sa'.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE 'sa' ACCOUNT

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Login current values
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Detailed | 
    Where-Object -Property SidString -eq '0x01' |
    Select-Object SqlInstance, Name, SidString | 
    #Format-Table -AutoSize;
    Format-Markdown
}
catch { throw $_ }

### Remediation:

**CIS Benchmark Query (T-SQL Reference):**
```sql
USE [master] 
GO 
ALTER LOGIN sa WITH NAME = '<renamed_sa>'';
GO
```

**dbatools PowerShell Command:**

In [ ]:
# UPDATE THE 'sa' ACCOUNT

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Set the Login to Disabled
    Rename-DbaLogin -SqlInstance $instances.SqlInstance -Login 'sa' -NewLogin '<renamed_sa>' -WarningAction SilentlyContinue |
    Format-Table -AutoSize;

    # Capture Login current values
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Detailed | 
    Where-Object -Property SidString -eq '0x01' |
    Select-Object SqlInstance, Name, SidString | 
    Format-Table -AutoSize;
}
catch { throw $_ }

## Additional Check: Ensure 'xp_cmdshell' Server Configuration Option is set to '0'

> **Note:** This check was included in the CIS SQL Server 2016 Benchmark but has been removed from the 2022 Benchmark. It is retained here as an additional security best practice.

> **Description:**
>
> The _xp_cmdshell_ option controls whether the _xp_cmdshell_ extended stored procedure can be used by an authenticated SQL Server user to execute operating-system command shell commands and return results as rows within the SQL client.

> **Rationale:**
>
> The _xp_cmdshell_ procedure is commonly used by attackers to read or write data to/from the underlying Operating System of a database server.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT name,
    CAST(value as int) as value_configured,
    CAST(value_in_use as int) as value_in_use
FROM sys.configurations
WHERE name = 'xp_cmdshell';
```

**Default Value:** By default, this option is disabled (0).

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'xp_cmdshell' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'XPCmdShellEnabled'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

### Remediation:

**CIS Benchmark Query (T-SQL Reference):**
```sql
-- Set the state of the option
EXECUTE sp_configure 'show advanced options', 1;
RECONFIGURE;
GO
EXECUTE sp_configure 'xp_cmdshell', 0;
RECONFIGURE;
GO
EXECUTE sp_configure 'show advanced options', 0;
RECONFIGURE;
GO
```

**dbatools PowerShell Command:**

In [ ]:
# UPDATE THE 'xp_cmdshell' VALUE TO '0'

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'XPCmdShellEnabled'
};

try {
    # Set the Advanced Option to False
    Set-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -Value $false -WarningAction SilentlyContinue |
    Format-Table -Property * -AutoSize;


    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, MinValue, MaxValue, DefaultValue, ConfiguredValue, RunningValue |
    Format-Table -Property * -AutoSize;
}
catch { throw $_ }

## 2.15 Ensure the 'AUTO_CLOSE' is set to 'OFF' on contained databases (Automated)

> **Benchmark Description:**
>
> _AUTO_CLOSE_ determines if a given database is closed or not after a connection terminates. If enabled, subsequent connections to the given database will require the database to be reopened and relevant procedure caches to be rebuilt.

> **Benchmark Rationale:**
>
> Because authentication of users for contained databases occurs within the database not at the server\instance level, the database must be opened every time to authenticate a user. The frequent opening/closing of the database consumes additional server resources and may contribute to a denial of service.

> **Possible Impact:**
>
> 

### Audit:

Use the following syntax to check for contained databases with AUTO_CLOSE enabled.

> No rows should be returned.

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT 
    name, containment, containment_desc, is_auto_close_on 
FROM sys.databases 
WHERE containment <> 0 and is_auto_close_on = 1;
```

**Default Value:**

By default, the database property _AUTO_CLOSE_ is _OFF_ which is equivalent to _is_auto_close_on_ = _0_.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'Auto Close' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Advanced Option current values
    Get-DbaDatabase -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, ContainmentType, AutoClose |
    Sort-Object -Property @{Expression = "SqlInstance"; Descending = $false }, @{Expression = "Name"; Descending = $false } |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

### Remediation:

**CIS Benchmark Query (T-SQL Reference):**
```sql
-- Set the state of the option
ALTER DATABASE <database_name> SET AUTO_CLOSE OFF;
```

**dbatools PowerShell Command:**

In [ ]:
# 2.16 AUTO_CLOSE remediation
# Uncomment and set your database name:
# Set-DbaDbState -SqlInstance $instances.SqlInstance -Database '<database_name>' -AutoClose $false

## 2.16 Ensure no login exists with the name 'sa' (Automated)

> **Benchmark Description:**
>
> The sa login (e.g. principal) is a widely known and often widely used SQL Server account. Therefore, there should not be a login called sa even when the original _sa_ login (_principal_id = 1_) has been renamed.

> **Benchmark Rationale:**
>
> Enforcing this control reduces the probability of an attacker executing brute force attacks against a well-known principal name.

> **Possible Impact:**
>
> It is not a good security practice to code applications or scripts to use the sa account. Given that it is a best practice to rename and disable the sa account, some 3rd party applications check for the existence of a login named sa and if it doesn't exist, creates one. Removing the sa login will prevent these scripts and applications from authenticating to the database server and executing required tasks or functions.

### Audit:

Use the following syntax to determine if a login named _sa_ exists.

> No rows should be returned.

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT principal_id, name
FROM sys.server_principals
WHERE name = 'sa';
```

**Default Value:**

The login with _principal_id = 1_ is named _sa_ by default.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE 'sa' ACCOUNT

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Login current values
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Detailed | 
    #Where-Object -Property ID -eq 1 |
    Where-Object -Property Name -eq "sa" |
    Select-Object SqlInstance, Name, ID, SidString | 
    #Format-Table -AutoSize;
    Format-Markdown
}
catch { throw $_ }

### Remediation:

**CIS Benchmark Query (T-SQL Reference):**
```sql
USE [master] 
GO 
ALTER LOGIN sa WITH NAME = '<renamed_sa>'';
GO
```

**dbatools PowerShell Command:**

In [ ]:
# UPDATE THE 'sa' ACCOUNT

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Set the Login to Disabled
    Rename-DbaLogin -SqlInstance $instances.SqlInstance -Login 'sa' -NewLogin '<renamed_sa>' -WarningAction SilentlyContinue |
    Format-Table -AutoSize;

    # Capture Login current values
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Detailed | 
    Where-Object -Property SidString -eq '0x01' |
    Select-Object SqlInstance, Name, SidString | 
    Format-Table -AutoSize;
}
catch { throw $_ }

## 2.17 Ensure 'clr strict security' Server Configuration Option is set to '1' (Automated)


> **Benchmark Description:**
>
> The _**clr strict security**_ option specifies whether the engine applies the `PERMISSION_SET` on assemblies in SQL Server.

> **Benchmark Rationale:**
>
> Enabling use of CLR assemblies widens the attack surface of SQL Server and puts it at risk from both inadvertent and malicious assemblies.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT name,
    CAST(value as int) as value_configured,
    CAST(value_in_use as int) as value_in_use
FROM sys.configurations
WHERE name = 'clr strict security';
```

> Both value columns must show `1` to be compliant.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT 'CLR Strict Security' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

# Define parameter splat
$params = @{
    Name = 'ClrStrictSecurity'
};

try {
    # Capture Advanced Option current values
    Get-DbaSpConfigure @params -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, DisplayName, DefaultValue, ConfiguredValue, RunningValue |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

# 3. Authentication and Authorization


> SQL Server authentication and authorization controls ensure that only authorized users can access the database engine and its data.

> **Benchmark Description:**
>
> Uses Windows Authentication to validate attempted connections.

> **Benchmark Rationale:**
>
> Windows provides a more robust authentication mechanism than SQL Server authentication.

> **Possible Impact:**
>
> Changing the login mode configuration requires a restart of the service.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT SERVERPROPERTY('IsIntegratedSecurityOnly') AS [login_mode];
```

> A `login_mode` of `1` indicates Windows Authentication Mode. A value of `0` indicates mixed mode.

**dbatools PowerShell Command:**

## 3.1 Ensure 'Server Authentication' Property is set to 'Windows Authentication Mode' (Automated)

In [ ]:
# CHECK THE CURRENT 'Server Authentication' VALUE

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Advanced Option current values
    Get-DbaInstanceProperty @params -SqlInstance $instances.SqlInstance -InstanceProperty LoginMode -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, Value |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

## 3.2 Ensure CONNECT permissions on the 'guest' user is Revoked within all SQL Server databases (Automated)


> **Benchmark Description:**
>
> Remove the right of the guest user to connect to SQL Server databases, except for master, msdb, and tempdb.

> **Benchmark Rationale:**
>
> A login assumes the identity of the guest user when it has access to SQL Server but not to a specific database. Revoking CONNECT for guest ensures logins cannot access databases without explicit permissions.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
USE <database_name>;
GO
SELECT DB_NAME() AS DatabaseName, 'guest' AS Database_User,
    [permission_name], [state_desc]
FROM sys.database_permissions
WHERE [grantee_principal_id] = DATABASE_PRINCIPAL_ID('guest')
    AND [state_desc] LIKE 'GRANT%'
    AND [permission_name] = 'CONNECT'
    AND DB_NAME() NOT IN ('master','tempdb','msdb');
```

> No rows should be returned to be compliant. Run per database.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE CURRENT CONNECT permissions on the 'guest' user

# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Advanced Option current values
    Get-DbaDbUser -SqlInstance $instances.SqlInstance -ExcludeDatabase master, msdb, tempdb, distribution -User 'guest' -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Database, Name, Login, LoginType, AuthenticationType, HasDbAccess |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

## 3.3 Ensure 'Orphaned Users' are Dropped From SQL Server Databases (Automated)


> **Benchmark Description:**
>
> A database user for which the corresponding SQL Server login is undefined or incorrectly defined cannot log in to the instance and is referred to as orphaned. These should be removed.

> **Benchmark Rationale:**
>
> Orphaned users should be removed to avoid potential misuse of those broken users.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
USE <database_name>;
GO
SELECT dp.type_desc, dp.sid, dp.name AS orphan_user_name,
    dp.authentication_type_desc
FROM sys.database_principals AS dp
    LEFT JOIN sys.server_principals AS sp ON dp.sid = sp.sid
WHERE sp.sid IS NULL
    AND dp.authentication_type_desc = 'INSTANCE';
```

> No rows should be returned. Run per database.

**dbatools PowerShell Command:**

In [ ]:
# Test server connections in case any connections are offline or invalid
$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    # Capture Advanced Option current values
    Get-DbaDbOrphanUser -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Database, Name, Login, LoginType, AuthenticationType, HasDbAccess |
    #Format-Table -Property * -AutoSize;
    Format-Markdown
}
catch { throw $_ }

## 3.4 Ensure SQL Authentication is not used in contained databases (Automated)


> **Benchmark Description:**
>
> Contained databases do not enforce password complexity rules for SQL Authenticated users.

> **Benchmark Rationale:**
>
> The absence of an enforced password policy may increase the likelihood of a weak credential being established in a contained database.

> **Possible Impact:**
>
> While contained databases provide flexibility in relocating databases, this must be balanced with the consideration that no password policy mechanism exists for SQL Authenticated users in contained databases.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT name AS DBUser
FROM sys.database_principals
WHERE name NOT IN ('dbo','Information_Schema','sys','guest')
    AND type IN ('U','S','G')
    AND authentication_type = 2;
```

> Run in each contained database. No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# CHECK FOR SQL AUTHENTICATION IN CONTAINED DATABASES
# Returns database-level users with SQL authentication in contained databases.
# If no contained databases exist, no results are returned.

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    $containedDbs = Get-DbaDatabase -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
        Where-Object { $_.ContainmentType -ne 'None' }

    if ($containedDbs) {
        foreach ($db in $containedDbs) {
            Get-DbaDbUser -SqlInstance $db.Parent.Name -Database $db.Name -WarningAction SilentlyContinue |
            Where-Object { $_.AuthenticationType -eq 'Database' } |
            Select-Object SqlInstance, Database, Name, LoginType, AuthenticationType
        } | Format-Markdown
    } else {
        Write-Output 'No contained databases found on any instance.'
    }
}
catch { throw $_ }

## 3.5 Ensure the SQL Server’s MSSQL Service Account is Not an Administrator (Manual)


> **Benchmark Description:**
>
> The service account and/or service SID used by the MSSQLSERVER service should not be a member of the Windows Administrators group either directly or indirectly. `LocalSystem` (`NT AUTHORITY\SYSTEM`) should not be used as it has higher privileges than the SQL Server service requires.

> **Benchmark Rationale:**
>
> Following the principle of least privilege, the service account should have no more privileges than required to do its job.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT *
FROM [master].[sys].[dm_server_registry]
WHERE registry_key = 'HKLM\SYSTEM\CurrentControlSet\Services\MSSQLSERVER'
    AND value_data = 'LocalSystem';
```

> No rows should be returned. Also verify the service account is not in the local Administrators group.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE SQL SERVER SERVICE ACCOUNTS
# Verify these accounts are NOT members of the local Administrators group.

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaService -ComputerName $instances.SqlInstance -WarningAction SilentlyContinue |
    Where-Object { $_.ServiceType -eq 'Engine' } |
    Select-Object ComputerName, ServiceType, ServiceName, StartName, State, StartMode |
    Format-Markdown
}
catch { throw $_ }

## 3.6 Ensure the SQL Server’s SQLAgent Service Account is Not an Administrator (Manual)


> **Benchmark Description:**
>
> The service account and/or service SID used by the SQLSERVERAGENT service should not be a member of the Windows Administrators group either directly or indirectly. `LocalSystem` should not be used as it has higher privileges than the SQL Agent service requires.

> **Benchmark Rationale:**
>
> Following the principle of least privilege, the service account should have no more privileges than required to do its job.

### **Audit:**

> Manual check. Verify the SQL Agent service account is not in the local Administrators group.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE SQL AGENT SERVICE ACCOUNTS
# Verify these accounts are NOT members of the local Administrators group.

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaService -ComputerName $instances.SqlInstance -WarningAction SilentlyContinue |
    Where-Object { $_.ServiceType -eq 'Agent' } |
    Select-Object ComputerName, ServiceType, ServiceName, StartName, State, StartMode |
    Format-Markdown
}
catch { throw $_ }

## 3.7 Ensure the SQL Server’s Full-Text Service Account is Not an Administrator (Manual)


> **Benchmark Description:**
>
> The service account and/or service SID used by the MSSQLFDLauncher service should not be a member of the Windows Administrators group either directly or indirectly. `LocalSystem` should not be used as it has higher privileges than the Full-Text service requires.

> **Benchmark Rationale:**
>
> Following the principle of least privilege, the service account should have no more privileges than required to do its job.

### **Audit:**

> Manual check. Verify the Full-Text service account is not in the local Administrators group.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE FULL-TEXT SERVICE ACCOUNTS
# Verify these accounts are NOT members of the local Administrators group.

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaService -ComputerName $instances.SqlInstance -WarningAction SilentlyContinue |
    Where-Object { $_.ServiceType -eq 'FullText' } |
    Select-Object ComputerName, ServiceType, ServiceName, StartName, State, StartMode |
    Format-Markdown
}
catch { throw $_ }

## 3.8 Ensure only the default permissions specified by Microsoft are granted to the public server role (Automated)


> **Benchmark Description:**
>
> Every SQL Server login belongs to the `public` server role. Only the default permissions specified by Microsoft should be granted to this role.

> **Benchmark Rationale:**
>
> Additional permissions granted to the `public` role are inherited by all logins, which could inadvertently grant excessive access.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT *
FROM master.sys.server_permissions
WHERE (grantee_principal_id = SUSER_SID(N'public')
    AND state_desc LIKE 'GRANT%')
    AND NOT (state_desc = 'GRANT' AND [permission_name] = 'VIEW ANY DATABASE' AND class_desc = 'SERVER')
    AND NOT (state_desc = 'GRANT' AND [permission_name] = 'CONNECT' AND class_desc = 'ENDPOINT' AND major_id = 2)
    AND NOT (state_desc = 'GRANT' AND [permission_name] = 'CONNECT' AND class_desc = 'ENDPOINT' AND major_id = 3)
    AND NOT (state_desc = 'GRANT' AND [permission_name] = 'CONNECT' AND class_desc = 'ENDPOINT' AND major_id = 4)
    AND NOT (state_desc = 'GRANT' AND [permission_name] = 'CONNECT' AND class_desc = 'ENDPOINT' AND major_id = 5);
```

> No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# 3.8 Check public server role permissions

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
SELECT * 
FROM master.sys.server_permissions 
WHERE (grantee_principal_id = SUSER_SID(N'public') 
and state_desc LIKE 'GRANT%') 
AND NOT (state_desc = 'GRANT' and [permission_name] = 'VIEW ANY DATABASE' and class_desc = 'SERVER') 
AND NOT (state_desc = 'GRANT' and [permission_name] = 'CONNECT' and class_desc = 'ENDPOINT' and major_id = 2) 
AND NOT (state_desc = 'GRANT' and [permission_name] = 'CONNECT' and class_desc = 'ENDPOINT' and major_id = 3) 
AND NOT (state_desc = 'GRANT' and [permission_name] = 'CONNECT' and class_desc = 'ENDPOINT' and major_id = 4) 
AND NOT (state_desc = 'GRANT' and [permission_name] = 'CONNECT' and class_desc = 'ENDPOINT' and major_id = 5);
"@

try {
    Invoke-DbaQuery -SqlInstance $instances.SqlInstance -Query $query -WarningAction SilentlyContinue |
    Format-Markdown
}
catch { throw $_ }

## 3.9 Ensure Windows BUILTIN groups are not SQL Logins (Automated)


> **Benchmark Description:**
>
> Windows `BUILTIN` groups (e.g., `BUILTIN\Administrators`) should not have SQL Server login access.

> **Benchmark Rationale:**
>
> BUILTIN groups provide broad Windows-level access that should not automatically translate to SQL Server access. Granting SQL logins to these groups violates the principle of least privilege.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT pr.[name], pe.[permission_name], pe.[state_desc]
FROM sys.server_principals pr
    JOIN sys.server_permissions pe ON pr.principal_id = pe.grantee_principal_id
WHERE pr.name LIKE 'BUILTIN%';
```

> No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# CHECK FOR WINDOWS BUILTIN GROUP LOGINS

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaLogin -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Where-Object { $_.Name -like 'BUILTIN\*' } |
    Select-Object SqlInstance, Name, LoginType, HasAccess, IsDisabled |
    Format-Markdown
}
catch { throw $_ }

## 3.10 Ensure Windows local groups are not SQL Logins (Automated)


> **Benchmark Description:**
>
> Windows local groups should not be granted SQL Server login access.

> **Benchmark Rationale:**
>
> Local groups can be modified by local administrators without SQL Server DBA knowledge, potentially granting unintended database access to unauthorized users.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT pr.[name] AS LocalGroupName, pe.[permission_name], pe.[state_desc]
FROM sys.server_principals pr
    JOIN sys.server_permissions pe ON pr.[principal_id] = pe.[grantee_principal_id]
WHERE pr.[type_desc] = 'WINDOWS_GROUP'
    AND pr.[name] LIKE CAST(SERVERPROPERTY('MachineName') AS nvarchar) + '%';
```

> No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# CHECK FOR WINDOWS LOCAL GROUP LOGINS

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaLogin -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Where-Object { $_.LoginType -eq 'WindowsGroup' -and $_.Name -like "$((Get-ComputerInfo).CsName)\*" } |
    Select-Object SqlInstance, Name, LoginType, HasAccess, IsDisabled |
    Format-Markdown
}
catch { throw $_ }

## 3.11 Ensure the public role in the msdb database is not granted access to SQL Agent proxies (Automated)


> **Benchmark Description:**
>
> The `public` role in the `msdb` database should not be granted access to SQL Agent proxies.

> **Benchmark Rationale:**
>
> Granting proxy access to the `public` role allows any database user to use the proxy, which may have elevated operating system credentials.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
USE [msdb]
GO
SELECT sp.name AS proxyname
FROM dbo.sysproxylogin spl
    JOIN sys.database_principals dp ON dp.sid = spl.sid
    JOIN sysproxies sp ON sp.proxy_id = spl.proxy_id
WHERE principal_id = USER_ID('public');
```

> No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# 3.11 Check public role access to SQL Agent proxies

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
SELECT sp.name AS proxyname 
FROM msdb.dbo.sysproxylogin spl 
JOIN sys.database_principals dp ON dp.sid = spl.sid 
JOIN msdb.dbo.sysproxies sp ON sp.proxy_id = spl.proxy_id 
WHERE principal_id = USER_ID('public');
"@

try {
    Invoke-DbaQuery -SqlInstance $instances.SqlInstance -Database 'msdb' -Query $query -WarningAction SilentlyContinue |
    Format-Markdown
}
catch { throw $_ }

## 3.12 Ensure the 'SYSADMIN' Role is Limited to Administrative or Built-in Accounts (Manual)


> **Benchmark Description:**
>
> The `sysadmin` server role should be limited to database administrators and designated built-in Microsoft accounts only.

> **Benchmark Rationale:**
>
> The `sysadmin` role has unrestricted access to all SQL Server resources. Limiting membership reduces the risk of unauthorized administrative actions.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT DISTINCT a.name, a.type_desc
FROM master.sys.server_principals a, sys.dm_server_services b
WHERE IS_SRVROLEMEMBER('sysadmin', a.name) = 1
    AND a.name = b.service_account;
```

> Review results to ensure only authorized accounts and built-in service accounts have sysadmin.

**dbatools PowerShell Command:**

In [ ]:
# CHECK SYSADMIN ROLE MEMBERSHIP

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaServerRoleMember -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Where-Object { $_.Role -eq 'sysadmin' -and $_.Name -notlike '##*' -and $_.Name -notlike 'NT SERVICE\*' } |
    Select-Object SqlInstance, @{ Name = 'LoginName'; Expression = { $_.Name } }, Role |
    Format-Markdown
}
catch { throw $_ }

## 3.13 Ensure no admin role membership in MSDB database (Automated)


> **Benchmark Description:**
>
> Membership in elevated `msdb` database roles (`db_owner`, `db_securityadmin`, `db_ddladmin`, `db_datawriter`) should be limited to authorized accounts only.

> **Benchmark Rationale:**
>
> Elevated roles in `msdb` grant the ability to modify or execute SQL Agent jobs, which could be used to escalate privileges or disrupt operations.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
USE [msdb]
GO
SELECT r.name AS RoleName, m.name AS MemberName
FROM sys.database_role_members AS drm
    INNER JOIN sys.database_principals AS r ON drm.role_principal_id = r.principal_id
    INNER JOIN sys.database_principals AS m ON drm.member_principal_id = m.principal_id
WHERE r.name IN ('db_owner', 'db_securityadmin', 'db_ddladmin', 'db_datawriter')
    AND m.name <> 'dbo';
```

> No rows should be returned (other than expected authorized accounts).

**dbatools PowerShell Command:**

In [ ]:
# CHECK ADMIN ROLE MEMBERSHIP IN MSDB

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaDbRoleMember -SqlInstance $instances.SqlInstance -Database msdb -WarningAction SilentlyContinue |
    Where-Object { $_.Role -in 'db_owner','db_securityadmin','db_ddladmin','db_datawriter' -and $_.UserName -ne 'dbo' } |
    Select-Object SqlInstance, Database, @{ Name = 'MemberName'; Expression = { $_.UserName } }, Role |
    Format-Markdown
}
catch { throw $_ }

# 4 Password Policies


> **Benchmark Description:**
>
> All SQL Authenticated logins should be created with the `MUST_CHANGE` option set to `ON` so the user is forced to change the password at first login.

> **Benchmark Rationale:**
>
> Enforcing password change on first login ensures that only the intended user knows the password, reducing the risk of credential sharing or interception during provisioning.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT name,
    CAST(LOGINPROPERTY(log.name, N'IsMustChange') AS bit) AS [MustChangePassword]
FROM sys.server_principals AS log
WHERE type = 'S'
    AND CAST(LOGINPROPERTY(log.name, N'IsMustChange') AS bit) = 1;
```

> No rows should be returned. Results indicate logins that haven't changed their initial password.

**dbatools PowerShell Command:**

## 4.1 Ensure 'MUST_CHANGE' Option is set to 'ON' for All SQL Authenticated Logins (Manual)

In [ ]:
# CHECK FOR LOGINS WITH MUST_CHANGE FLAG SET
# Logins returned here have not yet changed their password after being created with MUST_CHANGE = ON.

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Type SQL -Detailed -WarningAction SilentlyContinue |
    Where-Object { $_.IsMustChange -eq $true } |
    Select-Object SqlInstance, Name, IsMustChange |
    Format-Markdown
}
catch { throw $_ }

## 4.2 Ensure 'CHECK_EXPIRATION' Option is set to 'ON' for All SQL Authenticated Logins Within the Sysadmin Role (Automated)


> **Benchmark Description:**
>
> All SQL Authenticated logins that are members of the `sysadmin` role (or have `CONTROL SERVER` permission) should have `CHECK_EXPIRATION` set to `ON`.

> **Benchmark Rationale:**
>
> Sysadmin logins with non-expiring passwords present a persistent security risk if the password is compromised.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT l.[name], 'sysadmin membership' AS 'Access_Method'
FROM sys.sql_logins AS l
WHERE IS_SRVROLEMEMBER('sysadmin', name) = 1
    AND l.is_expiration_checked <> 1
UNION ALL
SELECT l.[name], 'CONTROL SERVER' AS 'Access_Method'
FROM sys.sql_logins AS l
    JOIN sys.server_permissions AS p ON l.principal_id = p.grantee_principal_id
WHERE p.type = 'CL' AND p.state IN ('G', 'W')
    AND l.is_expiration_checked <> 1;
```

> No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# CHECK FOR SYSADMIN SQL LOGINS WITHOUT CHECK_EXPIRATION

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    $sysadmins = (Get-DbaServerRoleMember -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
        Where-Object { $_.Role -eq 'sysadmin' }).Name

    Get-DbaLogin -SqlInstance $instances.SqlInstance -Type SQL -WarningAction SilentlyContinue |
    Where-Object { $_.Name -in $sysadmins -and -not $_.PasswordExpirationEnabled } |
    Select-Object SqlInstance, Name, PasswordExpirationEnabled |
    Format-Markdown
}
catch { throw $_ }

## 4.3 Ensure 'CHECK_POLICY' Option is set to 'ON' for All SQL Authenticated Logins (Automated)


> **Benchmark Description:**
>
> All SQL Authenticated logins should have `CHECK_POLICY` set to `ON` to enforce the Windows password complexity policy.

> **Benchmark Rationale:**
>
> Without password policy enforcement, SQL logins may be created with weak or easily guessable passwords.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT name, is_disabled FROM sys.sql_logins WHERE is_policy_checked = 0;
```

> No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# CHECK FOR SQL LOGINS WITHOUT CHECK_POLICY

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaLogin -SqlInstance $instances.SqlInstance -Type SQL -WarningAction SilentlyContinue |
    Where-Object { -not $_.PasswordPolicyEnforced } |
    Select-Object SqlInstance, Name, PasswordPolicyEnforced, IsDisabled |
    Format-Markdown
}
catch { throw $_ }

## 5.1 Ensure 'Maximum number of error log files' is set to greater than or equal to '12' (Automated)


> **Benchmark Description:**
>
> SQL Server error log files should be retained in sufficient number (12 or more) to support forensic analysis and troubleshooting.

> **Benchmark Rationale:**
>
> The SQL Server error log contains critical information about system events. Retaining too few log files risks losing important diagnostic data during log recycling.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
DECLARE @NumErrorLogs int;
EXEC master.sys.xp_instance_regread
    N'HKEY_LOCAL_MACHINE',
    N'Software\Microsoft\MSSQLServer\MSSQLServer',
    N'NumErrorLogs',
    @NumErrorLogs OUTPUT;
SELECT ISNULL(@NumErrorLogs, -1) AS [NumberOfLogFiles];
```

> Value should be `12` or greater. A value of `-1` indicates the default (6 files).

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE NUMBER OF ERROR LOG FILES
# CIS requires 12 or more.

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaErrorLogConfig -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, LogCount, LogSize |
    Format-Markdown
}
catch { throw $_ }

## 5.2 Ensure 'Default Trace Enabled' Server Configuration Option is set to '1' (Automated)


> **Benchmark Description:**
>
> The `default trace enabled` server configuration option provides a lightweight audit trail of database activity.

> **Benchmark Rationale:**
>
> Default trace provides valuable security-relevant information about login activity, permission changes, and schema modifications.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT name,
    CAST(value as int) as value_configured,
    CAST(value_in_use as int) as value_in_use
FROM sys.configurations
WHERE name = 'default trace enabled';
```

> Both value columns must show `1` to be compliant.

**dbatools PowerShell Command:**

In [ ]:
# CHECK THE 'Default Trace Enabled' VALUE

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaSpConfigure -SqlInstance $instances.SqlInstance -Name 'DefaultTraceEnabled' -WarningAction SilentlyContinue |
    Select-Object SqlInstance, DisplayName, ConfiguredValue, RunningValue |
    Format-Markdown
}
catch { throw $_ }

## 5.3 Ensure 'Login Auditing' is set to 'failed logins' (Automated)


> **Benchmark Description:**
>
> SQL Server login auditing should be configured to capture at least failed login attempts.

> **Benchmark Rationale:**
>
> Logging failed logins provides evidence of brute force or credential-guessing attacks against the SQL Server instance.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
EXEC xp_loginconfig 'audit level';
```

> The `config_value` should show `failure` or `all` to be compliant.

**dbatools PowerShell Command:**

In [ ]:
# 5.3 Check login auditing level

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
EXEC xp_loginconfig 'audit level';
"@

try {
    Invoke-DbaQuery -SqlInstance $instances.SqlInstance -Query $query -WarningAction SilentlyContinue |
    Format-Markdown
}
catch { throw $_ }

## 5.4 Ensure 'SQL Server Audit' is set to capture both 'failed' and 'successful logins' (Automated)


> **Benchmark Description:**
>
> SQL Server Audit should be configured with a server audit specification that captures both failed and successful login events.

> **Benchmark Rationale:**
>
> Capturing both failed and successful logins enables detection of unauthorized access attempts as well as forensic review of who accessed the system and when.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT
    S.name AS 'Audit Name',
    CASE S.is_state_enabled WHEN 1 THEN 'Y' WHEN 0 THEN 'N' END AS 'Audit Enabled',
    S.type_desc AS 'Write Location',
    SA.name AS 'Audit Specification Name',
    CASE SA.is_state_enabled WHEN 1 THEN 'Y' WHEN 0 THEN 'N' END AS 'Audit Specification Enabled',
    SAD.audit_action_name,
    SAD.audited_result
FROM sys.server_audit_specification_details AS SAD
    JOIN sys.server_audit_specifications AS SA ON SAD.server_specification_id = SA.server_specification_id
    JOIN sys.server_audits AS S ON SA.audit_guid = S.audit_guid
WHERE SAD.audit_action_id IN ('CNAU', 'LGFL', 'LGSD');
```

> Both audit and specification should be enabled, capturing `FAILED_LOGIN_GROUP` and `SUCCESSFUL_LOGIN_GROUP`.

**dbatools PowerShell Command:**

In [ ]:
# CHECK SQL SERVER AUDIT AND AUDIT SPECIFICATIONS

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Write-Host '=== Server Audits ===' -ForegroundColor Cyan
    Get-DbaInstanceAudit -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, Enabled, FilePath, QueueDelay |
    Format-Markdown

    Write-Host '=== Server Audit Specifications ===' -ForegroundColor Cyan
    Get-DbaInstanceAuditSpecification -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Name, Enabled, AuditName |
    Format-Markdown
}
catch { throw $_ }

## 6.1 Ensure Database and Application User Input is Sanitized (Manual)


> **Benchmark Description:**
>
> Database and application user input should always be sanitized using parameterized queries or stored procedures to prevent SQL injection attacks.

> **Benchmark Rationale:**
>
> SQL injection is one of the most common and dangerous attack vectors. Proper input sanitization prevents attackers from executing arbitrary SQL commands.

> This is a manual review item — it cannot be verified through SQL Server queries alone.

### **Audit:**

> This is a manual code review item. No SQL Server query can verify this.

**dbatools PowerShell Command:**

In [ ]:
# 6.1 Database and Application User Input Sanitization
# Manual check — review application code for parameterized queries
# and input validation. Not verifiable via SQL Server queries.

## 6.2 Ensure 'CLR Assembly Permission Set' is set to 'SAFE_ACCESS' for All CLR Assemblies (Automated)


> **Benchmark Description:**
>
> All user-created CLR assemblies should use the `SAFE_ACCESS` permission set. Assemblies with `UNSAFE` or `EXTERNAL_ACCESS` permission sets can access external system resources such as files, the network, and the registry.

> **Benchmark Rationale:**
>
> Assemblies with `UNSAFE` or `EXTERNAL_ACCESS` permissions can be used to access system resources outside of SQL Server, widening the attack surface.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
USE <database_name>;
GO
SELECT name, permission_set_desc
FROM sys.assemblies
WHERE is_user_defined = 1
    AND name <> 'Microsoft.SqlServer.Types';
```

> All assemblies should show `SAFE_ACCESS`. Run per database.

**dbatools PowerShell Command:**

In [ ]:
# 6.2 Check CLR Assembly permission sets
# NOTE: Run per database. Adjust -Database parameter as needed.

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
SELECT name, permission_set_desc 
FROM sys.assemblies 
WHERE is_user_defined = 1 AND name <> 'Microsoft.SqlServer.Types';
"@

try {
    # Run against each user database
    foreach ($inst in $instances.SqlInstance) {
        $dbs = Get-DbaDatabase -SqlInstance $inst -ExcludeSystem -WarningAction SilentlyContinue
        foreach ($db in $dbs) {
            Invoke-DbaQuery -SqlInstance $inst -Database $db.Name -Query $query -WarningAction SilentlyContinue
        }
    } | Format-Markdown
}
catch { throw $_ }

## 7.1 Ensure 'Symmetric Key encryption algorithm' is set to 'AES_128' or higher in non-system databases (Automated)


> **Benchmark Description:**
>
> Symmetric keys in non-system databases should use `AES_128` or higher encryption algorithms.

> **Benchmark Rationale:**
>
> Weaker encryption algorithms (e.g., DES, Triple DES) are vulnerable to brute force attacks. AES is the current industry standard for symmetric encryption.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT DB_NAME() AS Database_Name, name AS Key_Name, algorithm_desc
FROM sys.symmetric_keys
WHERE algorithm_desc NOT IN ('AES_128','AES_192','AES_256')
    AND db_id() > 4;
```

> No rows should be returned. Run per user database.

**dbatools PowerShell Command:**

In [ ]:
# 7.1 Check symmetric key encryption algorithms across all user databases

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
SELECT DB_NAME() AS Database_Name, name AS Key_Name, algorithm_desc
FROM sys.symmetric_keys
WHERE algorithm_desc NOT IN ('AES_128','AES_192','AES_256')
AND db_id() > 4;
"@

try {
    foreach ($inst in $instances.SqlInstance) {
        $dbs = Get-DbaDatabase -SqlInstance $inst -ExcludeSystem -WarningAction SilentlyContinue
        foreach ($db in $dbs) {
            Invoke-DbaQuery -SqlInstance $inst -Database $db.Name -Query $query -WarningAction SilentlyContinue
        }
    } | Format-Markdown
}
catch { throw $_ }

## 7.2 Ensure Asymmetric Key Size is set to 'greater than or equal to 2048' in non-system databases (Automated)


> **Benchmark Description:**
>
> Asymmetric keys in non-system databases should have a key length of 2048 bits or greater.

> **Benchmark Rationale:**
>
> Keys shorter than 2048 bits are considered weak and susceptible to attack with modern computing resources.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT DB_NAME() AS Database_Name, name AS Key_Name, key_length
FROM sys.asymmetric_keys
WHERE key_length < 2048
    AND db_id() > 4;
```

> No rows should be returned. Run per user database.

**dbatools PowerShell Command:**

In [ ]:
# 7.2 Check asymmetric key sizes across all user databases

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
SELECT DB_NAME() AS Database_Name, name AS Key_Name, key_length
FROM sys.asymmetric_keys
WHERE key_length < 2048
AND db_id() > 4;
"@

try {
    foreach ($inst in $instances.SqlInstance) {
        $dbs = Get-DbaDatabase -SqlInstance $inst -ExcludeSystem -WarningAction SilentlyContinue
        foreach ($db in $dbs) {
            Invoke-DbaQuery -SqlInstance $inst -Database $db.Name -Query $query -WarningAction SilentlyContinue
        }
    } | Format-Markdown
}
catch { throw $_ }

## 7.3 Ensure Database Backups are Encrypted (Automated)


> **Benchmark Description:**
>
> Database backups should be encrypted to protect data at rest.

> **Benchmark Rationale:**
>
> Unencrypted backups are vulnerable to theft and unauthorized access. If backup media is lost or stolen, the data can be read by anyone.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT b.key_algorithm, b.encryptor_type, d.is_encrypted,
    b.database_name, b.server_name, b.backup_finish_date
FROM msdb.dbo.backupset b
    INNER JOIN sys.databases d ON b.database_name = d.name
WHERE b.key_algorithm IS NULL
    AND b.encryptor_type IS NULL
    AND d.is_encrypted = 0;
```

> No rows should be returned.

**dbatools PowerShell Command:**

In [ ]:
# 7.3 Check unencrypted database backups

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
SELECT b.key_algorithm, b.encryptor_type, d.is_encrypted, 
    b.database_name, b.server_name, b.backup_finish_date 
FROM msdb.dbo.backupset b 
    INNER JOIN sys.databases d ON b.database_name = d.name 
WHERE b.key_algorithm IS NULL AND b.encryptor_type IS NULL AND d.is_encrypted = 0;
"@

try {
    Invoke-DbaQuery -SqlInstance $instances.SqlInstance -Query $query -WarningAction SilentlyContinue |
    Format-Markdown
}
catch { throw $_ }

## 7.4 Ensure Network Encryption is Configured and Enabled (Automated)


> **Benchmark Description:**
>
> Network connections to SQL Server should be encrypted to protect data in transit.

> **Benchmark Rationale:**
>
> Unencrypted network traffic can be intercepted and read using packet sniffing tools. Enforcing encryption protects against man-in-the-middle attacks and data exposure.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT DISTINCT encrypt_option
FROM sys.dm_exec_connections c
WHERE net_transport <> 'Shared memory'
    AND c.endpoint_id NOT IN (
        SELECT endpoint_id FROM sys.database_mirroring_endpoints
        WHERE encryption_algorithm IS NOT NULL
    );
```

> The `encrypt_option` should show `TRUE` for all rows.

**dbatools PowerShell Command:**

In [ ]:
# 7.4 Check network encryption

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$query = @"
SELECT DISTINCT encrypt_option 
FROM sys.dm_exec_connections c 
WHERE net_transport <> 'Shared memory' 
AND c.endpoint_id NOT IN (SELECT endpoint_id FROM sys.database_mirroring_endpoints WHERE encryption_algorithm IS NOT NULL);
"@

try {
    Invoke-DbaQuery -SqlInstance $instances.SqlInstance -Query $query -WarningAction SilentlyContinue |
    Format-Markdown
}
catch { throw $_ }

## 7.5 Ensure Databases are Encrypted with TDE (Automated)


> **Benchmark Description:**
>
> Transparent Data Encryption (TDE) should be enabled on all user databases to encrypt data at rest.

> **Benchmark Rationale:**
>
> TDE encrypts the database files at the storage level, protecting against unauthorized access to the physical database files or backup media.

### **Audit:**

**CIS Benchmark Query (T-SQL Reference):**

```sql
SELECT database_id, name, is_encrypted
FROM sys.databases
WHERE database_id > 4
    AND is_encrypted != 1;
```

> No rows should be returned. Results indicate unencrypted user databases.

**dbatools PowerShell Command:**

In [ ]:
# CHECK FOR DATABASES NOT ENCRYPTED WITH TDE

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaDbEncryption -SqlInstance $instances.SqlInstance -WarningAction SilentlyContinue |
    Select-Object SqlInstance, Database, EncryptionEnabled |
    Format-Markdown
}
catch { throw $_ }

## 8.1 Ensure 'SQL Server Browser Service' is configured correctly (Manual)


> **Benchmark Description:**
>
> The SQL Server Browser service should be disabled unless required to support named instances or multiple instances on the same server.

> **Benchmark Rationale:**
>
> The Browser service exposes instance names and connection details to the network. Disabling it reduces the attack surface by hiding instance discovery information from potential attackers.

### **Audit:**

> Manual check via SQL Server Configuration Manager or PowerShell.

**dbatools PowerShell Command:**

In [ ]:
# CHECK SQL SERVER BROWSER SERVICE STATUS

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

try {
    Get-DbaService -ComputerName $instances.SqlInstance -WarningAction SilentlyContinue |
    Where-Object { $_.ServiceType -eq 'Browser' } |
    Select-Object ComputerName, ServiceName, State, StartMode |
    Format-Markdown
}
catch { throw $_ }

---

# 9. Appendix — Establishing an Audit/Scan User

The CIS Benchmark recommends creating a dedicated least-privilege login for running audit queries. This avoids using `sysadmin` or `db_owner` for compliance scans.

The script below creates a Windows login and grants only the permissions needed to execute the CIS audit queries. It dynamically provisions the user in all online user databases, including databases in Always On Availability Groups.

> **Note:** Replace `DOMAIN\cis-scan` with your environment's service account or group. Re-run this script (or schedule it as an Agent job) whenever new databases are added.

**CIS Benchmark Query (T-SQL Reference):**

```sql
USE [master]
GO

-- Create the login if it doesn't exist
IF NOT EXISTS (SELECT * FROM sys.server_principals WHERE name = 'DOMAIN\cis-scan')
    CREATE LOGIN [DOMAIN\cis-scan] FROM WINDOWS WITH DEFAULT_DATABASE=[master]
GO

-- Grant server-level permissions
GRANT VIEW SERVER STATE TO [DOMAIN\cis-scan]
GO

-- Create user in master and grant xp_loginconfig execute
USE master
IF NOT EXISTS (SELECT * FROM sys.database_principals WHERE name = 'cis-scan')
    CREATE USER [cis-scan] FOR LOGIN [DOMAIN\cis-scan]
GO
GRANT EXECUTE ON sys.xp_loginconfig TO [cis-scan]
GO

-- Create user in msdb and grant proxy visibility
USE msdb
IF NOT EXISTS (SELECT * FROM sys.database_principals WHERE name = 'cis-scan')
    CREATE USER [cis-scan] FOR LOGIN [DOMAIN\cis-scan]
GO
GRANT SELECT ON dbo.sysproxies TO [cis-scan]
GRANT SELECT ON dbo.sysproxylogin TO [cis-scan]
GO
```

**dbatools PowerShell Command:**

In [ ]:
# 9. Establish CIS Audit/Scan User
# Replace 'DOMAIN\cis-scan' with your environment's service account

$instances = Test-DbaConnection -SqlInstance $SqlInstances.Instance -WarningAction SilentlyContinue
if ( $null -eq $instances ) {
    Write-Output "There are no valid servers with running SQL Instances. Please check on the status of the Instances.";
    return;
}

$scanLogin = 'DOMAIN\cis-scan'

foreach ($inst in $instances.SqlInstance) {
    Write-Host "Processing: $inst" -ForegroundColor Cyan

    # Create server login if it doesn't exist
    if (-not (Get-DbaLogin -SqlInstance $inst -Login $scanLogin -WarningAction SilentlyContinue)) {
        New-DbaLogin -SqlInstance $inst -Login $scanLogin -LoginType WindowsUser -WarningAction SilentlyContinue
        Write-Host "  Created login: $scanLogin" -ForegroundColor Green
    } else {
        Write-Host "  Login already exists: $scanLogin" -ForegroundColor Yellow
    }

    # Grant VIEW SERVER STATE
    Invoke-DbaQuery -SqlInstance $inst -Query "GRANT VIEW SERVER STATE TO [$scanLogin]" -WarningAction SilentlyContinue

    # Master: create user and grant xp_loginconfig
    $masterQuery = @"
        IF NOT EXISTS (SELECT * FROM sys.database_principals WHERE name = 'cis-scan')
            CREATE USER [cis-scan] FOR LOGIN [$scanLogin];
        GRANT EXECUTE ON sys.xp_loginconfig TO [cis-scan];
"@
    Invoke-DbaQuery -SqlInstance $inst -Database master -Query $masterQuery -WarningAction SilentlyContinue

    # msdb: create user and grant proxy visibility
    $msdbQuery = @"
        IF NOT EXISTS (SELECT * FROM sys.database_principals WHERE name = 'cis-scan')
            CREATE USER [cis-scan] FOR LOGIN [$scanLogin];
        GRANT SELECT ON dbo.sysproxies TO [cis-scan];
        GRANT SELECT ON dbo.sysproxylogin TO [cis-scan];
"@
    Invoke-DbaQuery -SqlInstance $inst -Database msdb -Query $msdbQuery -WarningAction SilentlyContinue

    # User databases: create user and grant SELECT on assemblies, symmetric keys, asymmetric keys
    $dbs = Get-DbaDatabase -SqlInstance $inst -ExcludeSystem -WarningAction SilentlyContinue
    foreach ($db in $dbs) {
        $dbQuery = @"
            IF NOT EXISTS (SELECT * FROM sys.database_principals WHERE name = 'cis-scan')
                CREATE USER [cis-scan] FOR LOGIN [$scanLogin];
            GRANT SELECT ON sys.assemblies TO [cis-scan];
            GRANT SELECT ON sys.symmetric_keys TO [cis-scan];
            GRANT SELECT ON sys.asymmetric_keys TO [cis-scan];
"@
        Invoke-DbaQuery -SqlInstance $inst -Database $db.Name -Query $dbQuery -WarningAction SilentlyContinue
    }
    Write-Host "  User databases provisioned: $($dbs.Count)" -ForegroundColor Green
}

Write-Host "`nDone. Verify the scan user:" -ForegroundColor Cyan
Get-DbaLogin -SqlInstance $instances.SqlInstance -Login $scanLogin |
Select-Object SqlInstance, Name, LoginType, HasAccess, IsDisabled |
Format-Markdown